# Butly LoCoMo Evaluation (Colab Pro)

This notebook is a **thin frontend**: it mounts Drive, prepares the repo and
two local model servers, then drives `python -m evals.locomo.cli`. All
evaluation logic lives in `evals/locomo/` — do not add scoring, replay, or
checkpoint code here.

**Two servers are required.** Butly's RAG needs real embeddings, and a chat
LLM cannot answer `/v1/embeddings`. So we run:

* a **chat** server (Qwen3-14B) on `CHAT_PORT` for chat/gatekeeper/summary/knowledge, and
* an **embedding** server (nomic-embed-text, started with `--embeddings`) on `EMBED_PORT`.

Prerequisites:

* Use a **GPU runtime** (Runtime -> Change runtime type -> GPU; L4/T4 is fine).
* Put the LoCoMo dataset JSON on Drive. Official data is CC BY-NC 4.0 and is
  **not** bundled with Butly — download it from
  https://github.com/snap-research/locomo yourself.
* Optionally add `HF_TOKEN` to Colab Secrets for gated/rate-limited downloads.

Artifacts (checkpoints included) are written to Drive, so a disconnected
runtime can continue with the **Resume** cell near the end.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- Parameters (edit these) ---
REPO_URL = 'https://github.com/unagisann/Butly.git'
BRANCH = 'main'
REPO_DIR = '/content/butly'

DRIVE_ROOT = '/content/drive/MyDrive/butly-evals'
DATASET_PATH = f'{DRIVE_ROOT}/data/locomo10.json'
RUN_ID = 'qwen3_14b_colab'   # one run directory per model + attempt

# Chat server (any OpenAI-compatible server works). Ports default high to
# dodge Colab's internal use of 8080.
CHAT_HF_REPO = 'Qwen/Qwen3-14B-GGUF'
CHAT_HF_FILE = 'Qwen3-14B-Q4_K_M.gguf'
CHAT_MODEL = 'qwen3-14b'
CHAT_PORT = 8090

# Embedding server (must be an embeddings-capable model)
EMBED_HF_REPO = 'nomic-ai/nomic-embed-text-v1.5-GGUF'
EMBED_HF_FILE = 'nomic-embed-text-v1.5.Q4_K_M.gguf'
EMBED_MODEL = 'nomic-embed-text'
EMBED_PORT = 8091

SAMPLE_LIMIT = 1
SESSION_LIMIT = 3
QUESTION_LIMIT = 10

In [ ]:
# --- Clone / update Butly and install dependencies ---
import os, subprocess
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
# --- Build llama.cpp once, cache binary + ALL shared libs on Drive ---
# The CUDA build is GPU-specific, so the cache key includes the GPU name.
# The server needs .so files that live outside build/bin (e.g.
# libllama-server-impl.so), so we collect every *.so* from the whole build
# tree (dereferenced: Drive cannot hold symlinks) and launch with
# LD_LIBRARY_PATH pointing at the restored directory.
import os, subprocess

gpu_name = subprocess.run(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
    capture_output=True, text=True,
).stdout.strip().replace(' ', '_') or 'unknown_gpu'
cache_dir = f'{DRIVE_ROOT}/bin/llama-v2-{gpu_name}'  # v2: old caches were incomplete
LOCAL_BIN = '/content/llama-bin'
SERVER_BIN = f'{LOCAL_BIN}/llama-server'

if os.path.isfile(f'{cache_dir}/llama-server'):
    print(f'using cached llama.cpp build for {gpu_name}')
    !mkdir -p {LOCAL_BIN} && cp -rL "{cache_dir}/." {LOCAL_BIN}/
    !chmod +x {SERVER_BIN}
else:
    print(f'no cache for {gpu_name}; building llama.cpp (several minutes)')
    !apt-get -qq install -y libcurl4-openssl-dev > /dev/null
    ![ -d /content/llama.cpp ] || git clone -q https://github.com/ggml-org/llama.cpp /content/llama.cpp
    !cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DLLAMA_CURL=OFF > /dev/null
    !cmake --build /content/llama.cpp/build --target llama-server -j > /dev/null
    !mkdir -p {LOCAL_BIN}
    !cp -L /content/llama.cpp/build/bin/llama-server {LOCAL_BIN}/
    !find /content/llama.cpp/build -name '*.so*' -exec cp -L {{}} {LOCAL_BIN}/ \;
    !mkdir -p "{cache_dir}" && cp -rL {LOCAL_BIN}/. "{cache_dir}/"
    print('cached build to', cache_dir)

# sanity check: the binary must resolve every shared library it needs
ldd_out = subprocess.run(
    ['ldd', SERVER_BIN], capture_output=True, text=True,
    env=dict(os.environ, LD_LIBRARY_PATH=LOCAL_BIN),
).stdout
missing = [line.strip() for line in ldd_out.splitlines() if 'not found' in line]
print('binary exists:', os.path.isfile(SERVER_BIN))
if missing:
    raise RuntimeError(
        f'unresolved libraries: {missing}; delete {cache_dir} and rerun this cell'
    )


In [ ]:
# --- Download both GGUF models (chat is not cached to Drive: HF is faster) ---
import os
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN')
    if tok:
        os.environ['HF_TOKEN'] = tok
except Exception:
    pass  # token only needed for gated / rate-limited downloads

!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download
chat_model_path = hf_hub_download(CHAT_HF_REPO, CHAT_HF_FILE)
embed_model_path = hf_hub_download(EMBED_HF_REPO, EMBED_HF_FILE)
print('chat  :', chat_model_path)
print('embed :', embed_model_path)

In [ ]:
# --- Robust server launcher: kill stale server on the port, log to file,
#     wait for /health, surface the log on failure ---
import os, subprocess, time, socket, pathlib, urllib.request

def _port_free(port):
    s = socket.socket()
    try:
        s.bind(('127.0.0.1', port)); return True
    except OSError:
        return False
    finally:
        s.close()

def start_llama_server(model_path, port, extra_args=None, timeout=600):
    extra_args = extra_args or []
    # free the port (kill a previous llama-server bound to it)
    subprocess.run(['pkill', '-9', '-f', f'--port {port}'], capture_output=True)
    for _ in range(15):
        if _port_free(port):
            break
        time.sleep(2)
    else:
        raise RuntimeError(f'port {port} is busy and did not free up; pick another in Parameters')

    log_path = f'/content/llama_{port}.log'
    log = open(log_path, 'w')
    # LOCAL_BIN holds the .so files collected at build time (Drive cache safe)
    env = dict(os.environ)
    env['LD_LIBRARY_PATH'] = LOCAL_BIN + ':' + env.get('LD_LIBRARY_PATH', '')
    proc = subprocess.Popen(
        [SERVER_BIN, '-m', model_path, '--port', str(port), '-ngl', '99'] + extra_args,
        stdout=log, stderr=subprocess.STDOUT, env=env,
    )
    for i in range(timeout // 2):
        if proc.poll() is not None:
            print(f'server on {port} EXITED with code', proc.returncode)
            print(pathlib.Path(log_path).read_text()[-3000:])
            raise RuntimeError(f'llama-server on {port} exited; see log above')
        try:
            urllib.request.urlopen(f'http://127.0.0.1:{port}/health', timeout=2)
            print(f'server on {port} is UP')
            return proc
        except Exception:
            if i % 15 == 14:
                tail = pathlib.Path(log_path).read_text().splitlines()
                print(f'  {port} loading...', tail[-1] if tail else '(no output yet)')
            time.sleep(2)
    raise RuntimeError(f'server on {port} did not become healthy in {timeout}s (see /content/llama_{port}.log)')

chat_server = start_llama_server(chat_model_path, CHAT_PORT)
embed_server = start_llama_server(
    embed_model_path, EMBED_PORT, extra_args=['--embeddings', '--pooling', 'mean']
)


In [ ]:
# --- Register both servers as Butly connections + write the eval profile ---
import json, pathlib, os
user_config = {
    'LLM_CONNECTIONS': [
        {
            'id': 'colab_local',
            'protocol': 'openai_compat',
            'base_url': f'http://127.0.0.1:{CHAT_PORT}/v1',
            'api_key_env': 'COLAB_LOCAL_API_KEY',
            'label': 'Colab chat server',
        },
        {
            'id': 'local_embedding',
            'protocol': 'openai_compat',
            'base_url': f'http://127.0.0.1:{EMBED_PORT}/v1',
            'api_key_env': 'COLAB_LOCAL_API_KEY',
            'label': 'Colab embedding server',
        },
    ]
}
pathlib.Path('user_config.json').write_text(json.dumps(user_config, indent=2))
os.environ['COLAB_LOCAL_API_KEY'] = 'local'  # llama.cpp accepts any key

profile = pathlib.Path('evals/locomo/profiles/full_local.example.yaml').read_text()
profile = profile.replace('qwen3-14b', CHAT_MODEL).replace('nomic-embed-text', EMBED_MODEL)
pathlib.Path('evals/locomo/profiles/full_local.yaml').write_text(profile)

# sanity check: both endpoints answer
import urllib.request
for name, port in [('chat', CHAT_PORT), ('embed', EMBED_PORT)]:
    body = urllib.request.urlopen(f'http://127.0.0.1:{port}/health', timeout=3).read().decode()
    print(f'{name} /health:', body)
print('connections + profile ready')

In [ ]:
# --- Run the evaluation (replay -> sleeptime -> QA -> score -> report) ---
!python -m evals.locomo.cli run \
  --dataset "{DATASET_PATH}" \
  --output-dir "{DRIVE_ROOT}/runs" \
  --run-id "{RUN_ID}" \
  --profile evals/locomo/profiles/full_local.yaml \
  --sample-limit {SAMPLE_LIMIT} \
  --session-limit {SESSION_LIMIT} \
  --question-limit {QUESTION_LIMIT}

In [ ]:
# --- Resume after a runtime disconnect (safe to re-run; skips finished work) ---
# Re-run the setup cells above first (mount, clone, build, download, servers,
# connections), then run this cell instead of the run cell.
!python -m evals.locomo.cli resume --run-dir "{DRIVE_ROOT}/runs/{RUN_ID}"

In [ ]:
# --- Show the summary ---
from IPython.display import Markdown, display
import pathlib
display(Markdown(pathlib.Path(f'{DRIVE_ROOT}/runs/{RUN_ID}/summary.md').read_text()))